In [0]:
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

# ---- Schema for the raw JSON in bronze.body ----
order_schema = StructType([
    StructField("order_id",         StringType()),
    StructField("user_id",          StringType()),
    StructField("product_id",       StringType()),
    StructField("product_category", StringType()),
    StructField("price",            DoubleType()),
    StructField("quantity",         IntegerType()),
    StructField("country",          StringType()),
    StructField("timestamp",        StringType()),
])

In [0]:
# ---- Read stream from Bronze ----
silver_stream = (
    spark.readStream
    .table("eh_streaming.oms.bronze_orders")
    .select(
        from_json(col("body"), order_schema).alias("d"),
        col("kafka_ts"),
    )
    .select("d.*", "kafka_ts")
    .withColumn("event_ts", to_timestamp("timestamp"))
    .withColumn("revenue", col("price") * col("quantity"))
    .filter(
        col("order_id").isNotNull() &
        (col("price") > 0) &
        (col("quantity") > 0)
    )
    .drop("timestamp")
)

silver_stream.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- kafka_ts: timestamp (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- revenue: double (nullable = true)



In [0]:
# ---- Write to Silver Delta table ----
(silver_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/eh_streaming/oms/checkpoints/silver")
    .trigger(processingTime="15 seconds")
    .toTable("eh_streaming.oms.silver_orders"))

In [0]:
# ---- Validate ----
print("Silver count:", spark.read.table("eh_streaming.oms.silver_orders").count())
display(spark.read.table("eh_streaming.oms.silver_orders").limit(5))

Silver count: 56966


order_id,user_id,product_id,product_category,price,quantity,country,kafka_ts,event_ts,revenue
ae21fa32-fa55-4fa7-8604-e33d5ff7de2e,0cb2c3e6-2bde-4bd7-a1f2-733f2cb0d560,PROD-005,books,19.99,4,BR,2026-05-27T04:45:09.007Z,2026-05-27T04:45:07.83086Z,79.96
e1347a2b-c7b1-4eb8-b930-dc3a85c31ca4,36e87659-6a4e-4b22-80cb-79c3c601b6a6,PROD-007,sports,75.0,1,DE,2026-05-27T04:45:09.007Z,2026-05-27T04:45:07.846827Z,75.0
a48e70ee-de5d-4742-ad5f-cb9c1416c47f,1a0d1ae8-668d-4b8c-a9b2-97e8ef021978,PROD-004,clothing,89.5,3,CA,2026-05-27T04:45:09.007Z,2026-05-27T04:45:07.8629Z,268.5
a3f5ae64-38e9-40f3-af82-62605e1cec43,3e98d1ac-dfbd-4d67-adf3-f48c258c36ca,PROD-006,home,159.0,2,DE,2026-05-27T04:45:09.007Z,2026-05-27T04:45:07.892832Z,318.0
7d366497-4359-4ef3-ae7a-d4aa78769844,89a49976-22be-41e0-a285-6f495f8c3daa,PROD-003,clothing,49.99,5,US,2026-05-27T04:45:09.007Z,2026-05-27T04:45:07.924304Z,249.95000000000002
